In [1]:
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

from myllia_metric import myllia_score

In [2]:
SVD_SEED = 6
MODEL_SEEDS = [6, 7, 8]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

EMB_DIM_PERT = 128
EMB_DIM_OUT  = 128
RANK_R = 32

DROPOUT = 0.10
LR = 2e-3
WD = 1e-4
EPOCHS = 400
BATCH_PERTS = 16
EVAL_EVERY = 25
PATIENCE = 12

GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

# loss weights
COS_BETA = 0.10

# regularization weights
UDELTA_L2 = 1e-4
SCALE_L2  = 1e-4
BIAS_L2   = 1e-5

means_path = "data/training_data_means.csv"
valmap_path = "data/pert_ids_val.csv"
sample_sub_path = "data/sample_submission.csv"

In [3]:
def score_delta(dt, dp):
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }

In [4]:
df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :]   # (80, 5127) deltas vs non-targeting
delta_baseline = D_train.mean(axis=0).astype(np.float32)

val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))

Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [5]:
val_targets = df_valmap["pert"].astype(str).tolist()

geneU = pd.Index([str(g).upper() for g in gene_columns])

X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SVD_SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)
k_svd = gene_emb_all.shape[1]
print("SVD k:", k_svd, "gene_emb_all:", gene_emb_all.shape)

SVD k: 80 gene_emb_all: (5127, 80)


In [7]:
gene2emb_pert = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_PERT].copy()
                 for i in range(len(gene_columns))}
gene2emb_out  = {gene_columns[i].upper(): gene_emb_all[i, :EMB_DIM_OUT].copy()
                 for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :EMB_DIM_PERT].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :EMB_DIM_OUT ].mean(axis=0).astype(np.float32)

def emb_pert(g: str) -> np.ndarray:
    return gene2emb_pert.get(str(g).upper(), emb_fallback_pert)

def emb_out(g: str) -> np.ndarray:
    return gene2emb_out.get(str(g).upper(), emb_fallback_out)

In [8]:
U_out = np.vstack([emb_out(g) for g in gene_columns]).astype(np.float32)    # (G, d_out)
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)  # (80, d_pert)

print("U_out:", U_out.shape, "Z_train:", Z_train.shape)

missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val = [g for g in val_targets if str(g).upper() not in geneU]
if missing_train:
    print("train perts not in gene_columns:", len(missing_train), "example:", missing_train[:12])
if missing_val:
    print("val perts not in gene_columns:", len(missing_val), "example:", missing_val[:12])

U_out: (5127, 80) Z_train: (80, 80)
train perts not in gene_columns: 8 example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8 example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']


In [10]:
Y = D_train.astype(np.float32)
N, G = Y.shape

Uo_t = torch.tensor(U_out, device=device)      # (G, d_out) base
Zt = torch.tensor(Z_train, device=device)    # (N, d_pert)
Yt = torch.tensor(Y, device=device)          # (N, G)

bias_init_t = torch.tensor(delta_baseline, device=device)  # (G,)

print("N:", N, "G:", G, "device:", device)

N: 80 G: 5127 device: cuda


In [11]:
def gate_smoothstep(x, a=GATE_A, b=GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def weighted_l1_like(delta_true, delta_pred, eps=EPS):
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)
    err = torch.abs(delta_pred - delta_true)
    num = torch.sum(w * err, dim=1)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)
    return torch.mean(num / den)

def weighted_cosine_loss(delta_true, delta_pred, eps=EPS):
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)
    dtw = w * delta_true
    dpw = w * delta_pred
    num = torch.sum(dtw * dpw, dim=1)
    den = torch.sqrt(torch.sum(dtw * dtw, dim=1) * torch.sum(dpw * dpw, dim=1) + eps)
    cos = num / den
    return torch.mean(1.0 - cos)

In [13]:
class BilinearDeltaModel(nn.Module):
    def __init__(self, d_pert, d_out, rank_r, dropout, G, bias_init):
        super().__init__()
        self.rank_r = rank_r

        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, 2 * rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * rank_r, rank_r),
            nn.LayerNorm(rank_r),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, 2 * rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * rank_r, rank_r),
            nn.LayerNorm(rank_r),
        )

        # trainable tweaks on output embeddings
        self.u_out_delta = nn.Parameter(torch.zeros(G, d_out))

        # per-gene scale and bias
        self.scale_gene = nn.Parameter(torch.ones(G))
        self.bias_gene  = nn.Parameter(bias_init.clone())
        self.bias_global = nn.Parameter(torch.zeros(1))

    def forward(self, z_pert, u_out_base):
        u = u_out_base + self.u_out_delta            # (G, d_out)
        p = self.proj_p(z_pert)                      # (B, R)
        o = self.proj_o(u)                           # (G, R)

        y = p @ o.T                                  # (B, G)
        y = y * self.scale_gene[None, :]             # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

def total_loss(model, dt, dp):
    l1 = weighted_l1_like(dt, dp)
    lc = weighted_cosine_loss(dt, dp)

    reg_u = torch.mean(model.u_out_delta * model.u_out_delta)
    reg_s = torch.mean((model.scale_gene - 1.0) ** 2)
    reg_b = torch.mean((model.bias_gene - bias_init_t) ** 2)

    return l1 + COS_BETA * lc + UDELTA_L2 * reg_u + SCALE_L2 * reg_s + BIAS_L2 * reg_b

In [15]:
def train_one_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT,
        G=G,
        bias_init=bias_init_t
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_PERTS):
            b = perm[start:start + BATCH_PERTS]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            loss = total_loss(model, Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        sched.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]
            s = score_delta(va_true, va_pred)
            sc = s["score"]

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_state

kf = KFold(n_splits=8, shuffle=True, random_state=SVD_SEED)
fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
    best_score, _ = train_one_fold(tr_idx, va_idx, seed=MODEL_SEEDS[0])
    fold_scores.append(float(best_score))
    print(f"fold {fold}: best_score={best_score:.6f}")
print(f"cv mean={float(np.mean(fold_scores)):.6f} std={float(np.std(fold_scores)):.6f}")

fold 1: best_score=0.035770
fold 2: best_score=0.061720
fold 3: best_score=0.042368
fold 4: best_score=0.058628
fold 5: best_score=0.081596
fold 6: best_score=0.073981
fold 7: best_score=0.070950
fold 8: best_score=0.034088
cv mean=0.057388 std=0.016961


In [16]:
def fit_full_model(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT,
        G=G,
        bias_init=bias_init_t
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_PERTS):
            b = perm[start:start + BATCH_PERTS]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            loss = total_loss(model, Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        sched.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t).detach().cpu().numpy().astype(np.float32)

            s = score_delta(Y, pred_np)
            sc = s["score"]
            print(f"[seed {seed}] epoch={epoch:4d} train_score={sc:.6f} wcos={s['wcos']:.6f} pred_wmae={s['pred_wmae']:.6f}")

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model


models = []
for sd in MODEL_SEEDS:
    models.append(fit_full_model(sd))

[seed 6] epoch=  25 train_score=0.041402 wcos=0.502425 pred_wmae=0.094119
[seed 6] epoch=  50 train_score=0.024637 wcos=0.449007 pred_wmae=0.097107
[seed 6] epoch=  75 train_score=0.070142 wcos=0.532050 pred_wmae=0.084100
[seed 6] epoch= 100 train_score=0.217281 wcos=0.648251 pred_wmae=0.073269
[seed 6] epoch= 125 train_score=0.387975 wcos=0.722496 pred_wmae=0.065364
[seed 6] epoch= 150 train_score=0.421574 wcos=0.750134 pred_wmae=0.063791
[seed 6] epoch= 175 train_score=0.574683 wcos=0.798262 pred_wmae=0.058026
[seed 6] epoch= 200 train_score=0.555099 wcos=0.800267 pred_wmae=0.058452
[seed 6] epoch= 225 train_score=0.679461 wcos=0.829173 pred_wmae=0.054283
[seed 6] epoch= 250 train_score=0.715959 wcos=0.837332 pred_wmae=0.052974
[seed 6] epoch= 275 train_score=0.739973 wcos=0.842820 pred_wmae=0.051993
[seed 6] epoch= 300 train_score=0.774422 wcos=0.850392 pred_wmae=0.050973
[seed 6] epoch= 325 train_score=0.779591 wcos=0.853168 pred_wmae=0.050814
[seed 6] epoch= 350 train_score=0.8012

In [17]:
def predict_delta_gene_ensemble(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb_pert(gene_symbol)[None, :].astype(np.float32), device=device)
    preds = []
    with torch.no_grad():
        for m in models:
            y = m(z, Uo_t).detach().cpu().numpy().astype(np.float32)[0]
            preds.append(y)
    return np.mean(np.stack(preds, axis=0), axis=0).astype(np.float32)

In [18]:
sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

idx = {g: i for i, g in enumerate(gene_columns)}
perm = [idx[g] for g in sub_gene_cols]

sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene_ensemble(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

out_path = "submission_bilinear_ensemble.csv"
sub.to_csv(out_path, index=False)
print("Wrote:", out_path, "| filled:", hit)

Wrote: submission_bilinear_ensemble.csv | filled: 60
